# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook provides an example for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the `mlcroissant` library, following the Croissant open standard.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the Croissant dataset (this downloads the schema and initializes the object)
dataset = mlc.Dataset(url)

# Access the dataset metadata
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"\nCroissant Dataset ID: {metadata.id}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s. This allows us to identify the data structures and how to address them when extracting from the dataset.

In [ ]:
# List the available record sets with their @id and fields
if hasattr(dataset, 'record_sets') and dataset.record_sets:
    print('All record sets in this dataset:')
    for recset in dataset.record_sets:
        print(f"- @id: {recset.id}, name: {getattr(recset, 'name', '<no name>')}")
        if hasattr(recset, 'fields'):
            for field in recset.fields:
                print(f"    - field @id: {field.id}, name: {getattr(field, 'name', '<no name>')}")
                # Optional: if columns/subfields available
                if hasattr(field, 'columns') and field.columns:
                    for column in field.columns:
                        print(f"        - column @id: {column.id}, name: {getattr(column, 'name', '<no name>')}")
else:
    print('No record sets were found in the metadata.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame using the record set and field `@id`s identified in the previous section.

In [ ]:
# Collect all record sets for extraction
record_sets = [recset.id for recset in getattr(dataset, 'record_sets', [])]
dataframes = {}

for recset_id in record_sets:
    try:
        records = list(dataset.records(record_set=recset_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[recset_id] = df
            print(f"Loaded {len(df)} records for record set: {recset_id}")
        else:
            print(f"No records found for record set: {recset_id}")
    except Exception as e:
        print(f"Failed to load records for record set {recset_id}: {e}")

if dataframes:
    # Just select the first loaded record set for demonstration
    example_record_set = list(dataframes.keys())[0]
    print(f"\nColumns in record set {example_record_set}:")
    print(dataframes[example_record_set].columns.tolist())
    print("\nFirst rows:")
    display(dataframes[example_record_set].head())
else:
    print('No data frames loaded from record sets.')

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering records, normalizing numeric fields, and grouping by a key attribute, referencing all fields by their `@id`.

In [ ]:
# In this example, we select a numeric field and a categorical group field using @id references found earlier.

if dataframes:
    # Use the first record set loaded in extraction step
    record_set_id = example_record_set
    df = dataframes[record_set_id]

    # Try to infer a numeric and a group field via heuristics (or specify manually if known)
    numeric_field = None
    group_field = None
    for col in df.columns:
        # Try to pick the first numeric-like column
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    for col in df.columns:
        if pd.api.types.is_categorical_dtype(df[col]) or df[col].dtype=='object':
            if col != numeric_field:
                group_field = col
                break
    if numeric_field:
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df[[numeric_field]].head())
        normalized_col = f"{numeric_field}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, normalized_col]].head())
        if group_field and group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped mean of filtered {numeric_field} by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric field detected for EDA.")
else:
    print("No dataframe available for EDA.")

## 5. Visualization
Visualize the numeric field and group field (if available) to better understand data distributions or relationships.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

- This notebook demonstrated loading, exploring, and extracting data from a Croissant-formatted dataset using the `mlcroissant` library.
- All entities (record sets, fields, columns) were referenced by their `@id` in code.
- The dataset contains ordered logistic regression results and related household survey analysis from Northern Kenya. Metadata review, record set overview, and programmatic data extraction allow flexible analysis, such as filtering, normalization, and grouping.
- Visualizations help in understanding distributions and group-level trends. For further exploration, consult the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273) documentation and schema.